# FLIR Cluster-Aware Splitting — Project Report

Particionamiento · evidencia local reproducible · evaluación exploratoria, sin entrenamiento del detector.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

root = Path.cwd()
report = root / "reports" / "splitting"
tables = report / "tables"
summary = json.loads((report / "summary.json").read_text(encoding="utf-8"))
runs = pd.read_csv(tables / "runs.csv")
candidates = pd.read_csv(tables / "clustering_candidates.csv")
final = pd.read_csv(tables / "final_candidates.csv")
sizes = pd.read_csv(tables / "sizes.csv")
classes = pd.read_csv(tables / "class_balance.csv")
variation = pd.read_csv(tables / "metric_variation.csv")
order = ["historical", "random_content", *candidates.candidate_label.tolist()]
pd.set_option("display.max_columns", 16)
pd.set_option("display.max_rows", 40)
def figure(name): display(Image(filename=str(report / "figures" / name), width=1150))
def show(table): display(table.round(4))
def ranges(columns):
    result = pd.DataFrame(index=order)
    for column in columns:
        values = runs.groupby("candidate_label")[column].agg(["min", "max"]).reindex(order)
        result[column] = [str(int(a)) if a == b else f"{int(a)}–{int(b)}" for a,b in values.to_numpy()]
    return result


## 1. Objetivo

Estudiar si mantener grupos correlacionados indivisibles reduce la correlación visual residual entre particiones, conservando tamaños y cobertura. Cada contenido exacto pesa una vez en las métricas visuales; los registros históricos y todas sus anotaciones pesan en el balance. Una reducción de correlación no demuestra por sí sola una mejora de generalización.

In [ ]:
display(Markdown(f"**{summary['run_count']} runs verificados**, {summary['clustering_candidate_count']} candidatos de clustering, {summary['valid_runs']} publicaciones válidas."))
targets = sizes.loc[sizes.candidate_label == "historical", ["split", "target_records"]].copy()
targets["target_ratio"] = targets.target_records / targets.target_records.sum()
show(targets)


## 2. Baselines

**Historical** conserva cada membership original, incluso contenidos presentes en varios splits. **Random content-level** permuta contenidos con semillas 0–4 y corta cerca de los targets ponderados por registros; no usa clases para el balance. **Cluster-aware** usa las mismas identidades exactas, más los grupos seleccionados y balance de clases. Esta diferencia de balance limita atribuir todos los cambios exclusivamente al clustering. No se ejecutó naive record-level random.

In [ ]:
show(runs.groupby("strategy").agg(runs=("split_space_id", "size"), contents=("content_count", "min"), records=("record_count", "min")))

## 3. Clustering candidates

Se seleccionan dos referencias por encoder × representación desde el Pareto existente: mínima fracción de ruido, seguida de diversidad maximin en rangos de ruido, número de grupos, ARI/AMI y coherencia visual/temporal. Se priorizan algoritmos ausentes. La regla no usa silhouette; conserva candidatos de utilidad todavía incierta, incluidos grupos dominantes. Los IDs corresponden a configuraciones, no a identidades de imágenes.

In [ ]:
show(candidates[["candidate_label", "encoder", "representation", "algorithm", "clustering_space_id", "noise_fraction", "n_clusters_excluding_noise", "parameters_common_clustered_ari_min", "visual_neighbor_coherence@10", "temporal_recall@5"]])

## 4. Noise policy

La política principal es **singleton**. Cada ruido conserva `cluster_id=-1` y recibe un grupo indivisible propio: no se inventa membership. Por ello, ruido visualmente similar puede cruzar splits. Similarity-components está reservado como ablación explícita basada en cuantiles del mismo encoder; no fue ejecutado.

In [ ]:
show(candidates[["candidate_label", "noise_fraction", "noise_points", "clustered_points"]])

## 5. Split construction protocol

Se utilizan targets de registros derivados del manifest. El MILP de SciPy/HiGHS asigna cantidades enteras de perfiles intercambiables a train/val/test y después expande grupos con semilla. Minimiza desviaciones L1 normalizadas de registros, cinco clases y vacíos. No recibe similitud ni temporalidad. Los grupos nunca se dividen. Se ejecutan cinco semillas; no se afirma unicidad. Un incumbente detenido por límite solo se acepta tras verificar todas las restricciones; su gap sigue visible. La política y formulación están en `docs/protocols/splitting.md`.

In [ ]:
show(runs.loc[runs.strategy == "cluster_aware"].groupby("candidate_label").agg(runs=("seed", "size"), optimal_within_tolerance=("solver_optimal", "sum"), maximum_gap=("solver_gap", "max")))

## 6. Split sizes

La figura compara tamaños de registros con los targets históricos. Las barras muestran media y rango entre semillas; las tablas conservan también contenidos únicos. La indivisibilidad puede hacer inalcanzable el tamaño deseado, aunque la asignación sea válida.

In [ ]:
figure("01_split_sizes_comparison.png")
show(ranges(["train_records", "val_records", "test_records", "train_contents", "val_contents", "test_contents"]))

## 7. Class balance

Cada instancia se cuenta desde su anotación histórica; los conflictos entre copias exactas se conservan. El heatmap muestra desviaciones absolutas de composición global en puntos porcentuales. Heavy Machinery se presenta explícitamente, al igual que labels vacíos, que no forman una sexta clase. Imágenes con presencia, contenidos e instancias por clase están en `tables/class_balance.csv`.

In [ ]:
figure("02_class_balance_comparison.png")
show(ranges(["train_heavy_machinery", "val_heavy_machinery", "test_heavy_machinery", "train_empty", "val_empty", "test_empty"]))
reference_labels = ["historical", "random_content", *final.candidate_label.tolist()]
show(classes.loc[classes.candidate_label.isin(reference_labels)].groupby(["candidate_label", "split", "class_name"])["instances"].mean().unstack("class_name"))

## 8. Exact duplicates

El histórico se audita con todas sus pertenencias. Para random content-level y cluster-aware, el verificador exige que cada contenido y todas sus ocurrencias permanezcan en un único split. Los pares de similitud excluyen self-content, de modo que este resultado no se mezcla con near-duplicates.

In [ ]:
figure("03_exact_duplicate_overlap.png")
show(runs.groupby("strategy").exact_duplicate_cross_split_count.agg(["min", "max"]))

## 9. Residual visual correlation — DINOv2

Para cada contenido se busca el vecino más similar de otro split en la **matriz original completa**. Se excluye self-content y se reporta un valor por contenido. La figura usa seed 0 de cada configuración, prefijada como representante; no son intervalos de confianza. En la tabla, las estadísticas se promedian entre las cinco semillas. Valores menores describen menor correlación residual bajo este encoder.

In [ ]:
figure("04_cross_split_nn_similarity_dinov2.png")
show(runs.groupby("candidate_label")[["dinov2_nn_mean", "dinov2_nn_median", "dinov2_nn_Q1", "dinov2_nn_Q3", "dinov2_nn_p90", "dinov2_nn_p95", "dinov2_nn_p99", "dinov2_nn_max", "dinov2_top1_cross_fraction", "dinov2_top5_cross_fraction", "dinov2_top10_cross_fraction", "dinov2_top20_cross_fraction"]].mean().reindex(order))

## 10. Residual visual correlation — CLIP

Se repite el cálculo de forma independiente con CLIP para todos los splits, incluidos los construidos desde DINOv2. Los valores absolutos de coseno no se comparan entre encoders. Los seis cohortes usan sus thresholds originales y retienen empates; son anidados. La figura de pares muestra fracciones, y la tabla mantiene cantidades y denominadores.

In [ ]:
figure("05_cross_split_nn_similarity_clip.png")
show(runs.groupby("candidate_label")[["clip_nn_mean", "clip_nn_median", "clip_nn_Q1", "clip_nn_Q3", "clip_nn_p90", "clip_nn_p95", "clip_nn_p99", "clip_nn_max", "clip_top1_cross_fraction", "clip_top5_cross_fraction", "clip_top10_cross_fraction", "clip_top20_cross_fraction"]].mean().reindex(order))
figure("06_high_similarity_cross_split_pairs.png")
quantiles = pd.read_csv(tables / "quantile_pairs.csv")
show(quantiles.loc[quantiles.candidate_label.isin(reference_labels)].groupby(["candidate_label", "encoder", "top_percentage"])[["pair_count", "cross_split_count", "cross_split_fraction"]].mean())

## 11. Temporal cross-split analysis

Las ventanas Δ≤1/5/10/25 se aplican a pares de contenidos distintos de la misma secuencia, con índice inferido por nombre. No son segundos ni timestamps verificados. La fragmentación completa de una secuencia es descriptiva: un video puede contener varias escenas. Con pocas secuencias, estas fracciones no permiten inferencia poblacional amplia.

In [ ]:
figure("07_temporal_cross_split_rates.png")
temporal = pd.read_csv(tables / "temporal.csv")
show(temporal.loc[temporal.candidate_label.isin(reference_labels)].groupby(["candidate_label", "frame_delta_max"])[["pair_count", "cross_split_count", "fraction"]].mean())
fragmentation = pd.read_csv(tables / "sequence_fragmentation.csv")
show(fragmentation.groupby("candidate_label")[["sequences", "fraction_1_splits", "fraction_2_splits", "fraction_3_splits"]].mean().reindex(order))

## 12. Cluster fracture

Se audita cada candidato contra historical, random y su propio cluster-aware. Noise no se agrupa bajo una etiqueta común para esta métrica. Los clusters usados como unidades deben tener exactamente cero fracturas; la proporción histórica y aleatoria es un diagnóstico post-hoc del mismo candidato.

In [ ]:
figure("08_cluster_fracture_comparison.png")
fracture = pd.read_csv(tables / "cluster_fracture.csv")
show(fracture.groupby(["candidate_label", "strategy"]).fracture_rate.mean().unstack("strategy"))

## 13. Historical vs random vs cluster-aware

Esta tabla reúne balance y correlación residual para mantener visibles los compromisos. Valores medios entre semillas; tablas locales contienen cada run. La comparación principal del efecto más allá de duplicados exactos es contra random content-level. El histórico no se fuerza a una sola pertenencia por contenido.

In [ ]:
show(runs.groupby("candidate_label")[["max_relative_record_deviation", "class_deviation_pp", "dinov2_nn_mean", "clip_nn_mean", "dinov2_top001_pairs", "clip_top001_pairs", "temporal_at5"]].mean().reindex(order))

## 14. Robustness across split seeds

Se comparan los nombres fijos train/val/test: no se permutan para mejorar acuerdos ni se usa ARI como criterio principal. Diez pares de semillas por candidato. El std poblacional resume cinco asignaciones, no un intervalo de confianza ni una replicación independiente del dataset.

In [ ]:
stability = pd.read_csv(tables / "seed_stability.csv")
show(stability.groupby("candidate_label").same_named_split_fraction.agg(["mean", "min", "max"]))
show(variation.loc[variation.strategy == "random_content", ["metric", "n", "mean", "std_population", "min", "max"]])

## 15. Pareto shortlist

La elegibilidad exige todas las semillas válidas, cero exact overlap/fracture, cinco clases en cada split y desviación relativa de registros ≤10%. Se minimizan los peores resultados entre semillas de balance, similitud de ambos encoders, pares top 0.1%, temporal Δ≤5 y variabilidad residual. No hay score ponderado. La figura muestra solo una proyección de los ocho criterios; no define el frente por sí sola.

In [ ]:
figure("09_split_candidate_pareto.png")
pareto = pd.read_csv(tables / "pareto.csv")
show(pareto[["candidate_label", "eligible", "pareto", "all_classes_covered", "max_relative_record_deviation", "class_deviation_pp", "dinov2_top001_fraction", "clip_top001_fraction", "residual_seed_std_max"]])

## 16. Final candidate splits

Se conservan hasta tres anclas del frente: mínimos del peor top 0.1% de cada encoder y de desviación de clases. Si coinciden, no se inventan recomendaciones adicionales. Seed 0 es siempre la representante, elegida antes de observar las métricas. Los resultados justifican candidatos exploratorios para revisar el protocolo del detector, no un encoder o clustering ganador universal.

In [ ]:
show(final[["candidate_label", "clustering_space_id", "representative_split_space_id", "final_reason"]])
show(runs.loc[runs.split_space_id.isin(final.representative_split_space_id), ["candidate_label", "split_space_id", "train_records", "val_records", "test_records", "train_heavy_machinery", "val_heavy_machinery", "test_heavy_machinery", "train_empty", "val_empty", "test_empty", "dinov2_top001_pairs", "clip_top001_pairs"]])
historical = runs.loc[runs.strategy == "historical"].iloc[0]
random_mean = runs.loc[runs.strategy == "random_content", "temporal_at5"].mean()
for row in runs.loc[runs.split_space_id.isin(final.representative_split_space_id)].itertuples():
    display(Markdown(f"**{row.candidate_label}**: temporal Δ≤5 = **{100*row.temporal_at5:.2f}%**, frente a historical **{100*historical.temporal_at5:.2f}%** y random **{100*random_mean:.2f}%** (media)."))
    if row.temporal_at5 > historical.temporal_at5:
        display(Markdown("Esta representante **no mejora el criterio temporal frente al histórico**. La selección conserva un compromiso, no una mejora uniforme."))
    if row.dinov2_nn_mean >= historical.dinov2_nn_mean or row.clip_nn_mean >= historical.clip_nn_mean:
        display(Markdown("Tampoco reduce la media NN frente al histórico en ambos encoders. Su papel como ancla de balance no demuestra mejor control global de correlación."))


## 17. Limitations

- La coherencia temporal es inferida; no existen timestamps verificados.
- Los espacios de similitud participaron en el análisis y la selección de clustering: esta revisión no es una validación externa independiente, aunque el solver no optimice sus métricas.
- Random no balancea clases; una ablación adicional de singleton balanceado ayudaría a aislar la contribución del agrupamiento.
- Se preservan conflictos de anotación, la multiplicidad de registros y la posibilidad de sesgo por esas repeticiones en un entrenamiento posterior.
- Singleton noise puede conservar relaciones intensas entre splits.
- Un incumbente factible no es necesariamente óptimo; se reporta la brecha del solver.
- La revisión visual anterior cubrió ejemplos limitados; este reporte no certifica semánticamente cada escena.
- No se ejecutaron similarity-components, materialización de imágenes ni YOLO. No se atribuye causalidad de leakage a la proximidad temporal.

## 18. Next step

Diseñar la comparación controlada del detector sobre historical, random content-level y los candidatos finales, con presupuesto, configuración, métricas y semillas comunes. Revisar los grupos seleccionados y los conflictos de anotación antes de materializar datos. El mecanismo de exportación solo referencia imágenes previamente materializadas y no copia ni mueve archivos. Precision, Recall, mAP@50 y mAP@50–95 siguen pendientes.